# 07 — Integrator comparison: GSL vs Filon (vs GPU)

Runs the same BubbleMaster setup through multiple integrator backends and compares
the resulting GW spectra and wall-clock times.

**Workflow**
1. Run **Section 1** to write the shared setup file.
2. Run **Section 2** to generate SLURM scripts (one per backend).
3. Submit all scripts: `! for s in scripts/compare_*.sh; do sbatch $s; done`
4. After jobs complete, run **Sections 3–5** to compare spectra and timing.

To add the GPU backend later: uncomment the `gpu` entry in `BACKENDS` below.

In [ ]:
import sys
import re
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import _cutoff_times, BubbleMasterParams

config = pt.Config()
REPO_ROOT = Path(pt.__file__).parent.parent

# ---------------------------------------------------------------------------
# Validation point
# ---------------------------------------------------------------------------
LAMBDA_BAR = 0.84
D          = 55.55

params = BubbleMasterParams(dz=0.005, n_w=64, n_k=51,
                             how_often_ds=5, baby_steps=100, cutoff_type=0)

model  = pt.PhysicsModel(pt.Phi4Potential(LAMBDA_BAR), config)
t_coll = float(model.kinematics.collision_time(D))
gamma  = float(model.kinematics.Gamma(t_coll))
TAG    = f'lb{LAMBDA_BAR}_g{gamma:.4g}'

t_0, t_cut, t_m, t_max = _cutoff_times(D, params.cutoff_type, params.t_0_scal)
TIMES = np.array([t_m])

RUN_DIR     = model.results_dir / 'integrator_compare' / TAG
SETUP_PATH  = RUN_DIR / 'setup.h5'
SCRIPTS_DIR = REPO_ROOT / 'scripts'
LOGS_DIR    = REPO_ROOT / 'logs'
RUN_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

print(f'TAG      = {TAG}')
print(f'd        = {D}   gamma = {gamma:.4f}')
print(f't_coll   = {t_coll:.3f}')
print(f't_cut    = {t_cut:.3f}   t_m = {t_m:.3f}   t_max = {t_max:.3f}')
print(f'run_dir  = {RUN_DIR}')

# ---------------------------------------------------------------------------
# Comparison selection — pick which convergence runs to use in Section 5.
# Both values must appear in FILON_N_VALUES / GSL_LIMIT_VALUES below.
# ---------------------------------------------------------------------------
COMPARE_FILON_N   = 2048
COMPARE_GSL_LIMIT = 1000

# ---------------------------------------------------------------------------
# Convergence study backends
# Filon: vary N_min panels.  GSL: vary subinterval limit.
# ---------------------------------------------------------------------------
FILON_N_VALUES   = [256, 512, 1024, 2048, 4096, 8192]
GSL_LIMIT_VALUES = [50, 100, 200, 500, 1000, 5000]

_filon_colors = plt.cm.Oranges(np.linspace(0.35, 0.95, len(FILON_N_VALUES)))
_gsl_colors   = plt.cm.Blues (np.linspace(0.35, 0.95, len(GSL_LIMIT_VALUES)))

def _filon_time(n):
    minutes = max(30, int(np.ceil((7.5 * n / 2048 + 15) / 30)) * 30)
    return f'{minutes // 60:02d}:{minutes % 60:02d}:00'

CONV_BACKENDS = [
    dict(name=f'filon_n{n}',
         label=f'Filon N={n}',
         binary=config.bubblemaster_filon_bin,
         out_dir=RUN_DIR / f'out_filon_n{n}',
         param=n,
         color=_filon_colors[i],
         nodes=1, cpus=128, time=_filon_time(n))
    for i, n in enumerate(FILON_N_VALUES)
] + [
    dict(name=f'gsl_l{lim}',
         label=f'GSL LIMIT={lim}',
         binary=config.bubblemaster_bin,
         out_dir=RUN_DIR / f'out_gsl_l{lim}',
         param=lim,
         color=_gsl_colors[i],
         nodes=1, cpus=128, time='02:00:00')
    for i, lim in enumerate(GSL_LIMIT_VALUES)
]

# GPU — standalone (no convergence sweep)
GPU_BACKEND = dict(name='gpu',
                   label='Filon (GPU, A100)',
                   binary=config.bubblemaster_gpu_bin,
                   out_dir=RUN_DIR / 'out_gpu',
                   color='#009E73',
                   nodes=1, cpus=32, time='00:30:00')

for b in CONV_BACKENDS:
    b['out_dir'].mkdir(parents=True, exist_ok=True)
GPU_BACKEND['out_dir'].mkdir(parents=True, exist_ok=True)

print(f'  {len(CONV_BACKENDS)} convergence backends')
print(f'  Comparison (Section 5): Filon N={COMPARE_FILON_N}  vs  GSL LIMIT={COMPARE_GSL_LIMIT}')

## 1. Write shared setup file

In [ ]:
pt.write_2d_setup(model, gamma=gamma, times=TIMES,
                  path=SETUP_PATH, params=params)
print(f'Written: {SETUP_PATH}')

## 2. Generate SLURM scripts

In [ ]:
def make_slurm(b):
    """Return SLURM script text for backend dict b."""
    is_gpu     = b['name'] == 'gpu'
    param_flag = f' --param {b["param"]}' if 'param' in b else ''

    if is_gpu:
        resource_block = f"""\
#SBATCH --constraint=gpu
#SBATCH --qos=shared
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task={b['cpus']}
#SBATCH --gpus-per-task=1"""
    else:
        resource_block = f"""\
#SBATCH --constraint=cpu
#SBATCH --qos=premium
#SBATCH --nodes={b['nodes']}
#SBATCH --tasks-per-node=1
#SBATCH --cpus-per-task={b['cpus']}"""

    return f"""#!/bin/bash
{resource_block}
#SBATCH --time={b['time']}
#SBATCH --job-name=bm_{TAG}_{b['name']}
#SBATCH --output={LOGS_DIR}/bm_{TAG}_{b['name']}_%j.out

export OMP_NUM_THREADS={b['cpus']}
export OMP_PLACES=threads
export OMP_PROC_BIND=spread

mkdir -p {b['out_dir']}
/usr/bin/time --verbose srun {b['binary']} {SETUP_PATH} {b['out_dir']}/{param_flag} --save-fields
"""

# --- Convergence study ---
print('=== Convergence study scripts ===')
for b in CONV_BACKENDS:
    p = SCRIPTS_DIR / f'conv_{TAG}_{b["name"]}.sh'
    p.write_text(make_slurm(b))
    print(f"  [{b['name']}]  N/LIMIT={b['param']}  t={b['time']}  {p}")

print()
print('Submit:')
for b in CONV_BACKENDS:
    print(f"  ! sbatch scripts/conv_{TAG}_{b['name']}.sh")

# --- GPU ---
print()
print('=== GPU script ===')
p = SCRIPTS_DIR / f'compare_{TAG}_{GPU_BACKEND["name"]}.sh'
p.write_text(make_slurm(GPU_BACKEND))
print(f"  {p}")
print(f"  ! sbatch scripts/compare_{TAG}_{GPU_BACKEND['name']}.sh")

## 3. Load results

Reads `result_*.h5` files from each backend's output directory and interpolates
to `t_m`.  Jobs that have not finished will print a warning and be skipped.

In [ ]:
def load_result(setup_path, out_dir):
    """Load spectrum at t_m.  Returns (w, spectrum, t_m)."""
    with h5py.File(setup_path, 'r') as f:
        t_m_setup = float(f.attrs['t_m'])
        times_all = f['times'][:]

    entries = []
    for p in Path(out_dir).glob('result_*.h5'):
        i_t = int(p.stem.split('_')[1])
        if i_t < len(times_all):
            entries.append((times_all[i_t], p))
    if not entries:
        raise FileNotFoundError(f'No result_*.h5 in {out_dir}')
    entries.sort()

    if len(entries) == 1:
        with h5py.File(entries[0][1], 'r') as f:
            return f['w'][:], f['spectrum'][:], t_m_setup

    t_avail = np.array([e[0] for e in entries])
    if t_m_setup < t_avail[0] or t_m_setup > t_avail[-1]:
        raise RuntimeError(
            f't_m={t_m_setup:.4f} outside available range '
            f'[{t_avail[0]:.4f}, {t_avail[-1]:.4f}] '
            f'({len(entries)}/{len(times_all)} steps done)'
        )

    i = max(0, int(np.searchsorted(t_avail, t_m_setup)) - 1)
    i = min(i, len(entries) - 2)
    alpha = (t_m_setup - t_avail[i]) / (t_avail[i+1] - t_avail[i])

    with h5py.File(entries[i][1], 'r') as f:
        w      = f['w'][:]
        spec_a = f['spectrum'][:]
    with h5py.File(entries[i+1][1], 'r') as f:
        spec_b = f['spectrum'][:]

    return w, spec_a + alpha * (spec_b - spec_a), t_m_setup


conv_results = {}
for b in CONV_BACKENDS:
    try:
        w, spec, t_m_val = load_result(SETUP_PATH, b['out_dir'])
        conv_results[b['name']] = dict(w=w, spec=spec, t_m=t_m_val, **b)
        print(f"[{b['name']:20s}]  loaded  param={b['param']}")
    except (FileNotFoundError, RuntimeError) as e:
        print(f"[{b['name']:20s}]  NOT READY: {e}")

try:
    w, spec, t_m_val = load_result(SETUP_PATH, GPU_BACKEND['out_dir'])
    conv_results['gpu'] = dict(w=w, spec=spec, t_m=t_m_val, **GPU_BACKEND)
    print(f"[{'gpu':20s}]  loaded")
except (FileNotFoundError, RuntimeError) as e:
    print(f"[{'gpu':20s}]  NOT READY: {e}")

print(f'\n{len(conv_results)}/{len(CONV_BACKENDS) + 1} jobs done')

## 4. Wall-clock timing

Reads `Maximum wall clock time` from the `/usr/bin/time --verbose` output in the
SLURM log files.

In [ ]:
def parse_walltime(log_dir, job_tag):
    """Return wall-clock seconds from the most recent matching SLURM log."""
    logs = sorted(Path(log_dir).glob(f'*{job_tag}*.out'))
    if not logs:
        return None
    text = logs[-1].read_text(errors='replace')
    m = re.search(r'Elapsed \(wall clock\) time.*?:\s+(\S+)', text)
    if not m:
        return None
    parts = [float(x) for x in m.group(1).split(':')]
    if len(parts) == 2:
        return parts[0]*60 + parts[1]
    return parts[0]*3600 + parts[1]*60 + parts[2]


print(f'{"Backend":<22}  {"Wall time":>12}')
print('-' * 38)
for name in conv_results:
    secs = parse_walltime(LOGS_DIR, f'{TAG}_{name}')
    if secs is not None:
        print(f"{name:<22}  {secs/60:>10.1f} min")
    else:
        print(f"{name:<22}  {'(log not found)':>12}")

## 5. Spectrum comparison

In [ ]:
filon_key = f'filon_n{COMPARE_FILON_N}'
gsl_key   = f'gsl_l{COMPARE_GSL_LIMIT}'

missing = [k for k in (filon_key, gsl_key) if k not in conv_results]
if missing:
    print(f'Not loaded yet: {missing} — run Section 3 after jobs complete.')
else:
    compare_keys = [gsl_key, filon_key] + (['gpu'] if 'gpu' in conv_results else [])
    compare = {k: conv_results[k] for k in compare_keys}

    fig, (ax_spec, ax_diff) = plt.subplots(
        2, 1, figsize=(7, 7),
        gridspec_kw=dict(height_ratios=[2, 1], hspace=0.05),
        sharex=True,
    )

    ref = compare[gsl_key]

    for name, r in compare.items():
        secs = parse_walltime(LOGS_DIR, f'{TAG}_{name}')
        timing_str = f'  [{secs/60:.1f} min]' if secs else ''
        label = r['label'] + timing_str

        ls = '-' if name == gsl_key else '--'
        ax_spec.plot(r['w'] * D, r['spec'], color=r['color'], lw=1.8,
                     linestyle=ls, label=label)

        if name != gsl_key:
            spec_interp = np.interp(ref['w'], r['w'], r['spec'])
            rel_diff    = (spec_interp - ref['spec']) / (np.abs(ref['spec']) + 1e-30)
            ax_diff.plot(ref['w'] * D, rel_diff * 100,
                         color=r['color'], lw=1.5,
                         label=f"{r['label']} vs {ref['label']}")

    ax_spec.set_ylabel(r'$\Omega_\mathrm{GW}$ spectrum  [a.u.]')
    ax_spec.set_title(
        rf'$\bar\lambda={LAMBDA_BAR}$   $d={D}$   $\gamma={gamma:.3f}$   $t_m={ref["t_m"]:.3f}$',
        fontsize=9
    )
    ax_spec.legend(frameon=False, fontsize=8)
    ax_spec.set_yscale('log')
    ax_spec.set_xscale('log')

    ax_diff.axhline(0, color='grey', lw=0.8, ls=':')
    ax_diff.set_xlabel(r'$k R_*$')
    ax_diff.set_ylabel(r'relative diff  [%]')
    ax_diff.legend(frameon=False, fontsize=7)

    plt.tight_layout()
    plt.savefig(REPO_ROOT / f'compare_integrators_{TAG}.pdf', bbox_inches='tight')
    plt.show()

## 6. Per-frequency-bin residual table

In [ ]:
filon_key = f'filon_n{COMPARE_FILON_N}'
gsl_key   = f'gsl_l{COMPARE_GSL_LIMIT}'

missing = [k for k in (filon_key, gsl_key) if k not in conv_results]
if missing:
    print(f'Need {missing} loaded — run Section 3 first.')
else:
    ref    = conv_results[gsl_key]
    others = [k for k in (filon_key, 'gpu') if k in conv_results]

    header = f"{'w':>10}  {'kR':>10}  {gsl_key:>18}"
    for n in others:
        header += f"  {n:>18}  {'rel diff [%]':>14}"
    print(header)
    print('-' * len(header))

    for w_j, s_ref in zip(ref['w'], ref['spec']):
        row = f"{w_j:10.4f}  {w_j*D:10.4f}  {s_ref:18.6e}"
        for n in others:
            r   = conv_results[n]
            s_n = np.interp(w_j, r['w'], r['spec'])
            rd  = (s_n - s_ref) / (abs(s_ref) + 1e-30) * 100
            row += f"  {s_n:18.6e}  {rd:+14.4f}"
        print(row)

## 7. Convergence study

In [ ]:
if not conv_results:
    print('No convergence results loaded yet.')
else:
    filon_res = {k: v for k, v in conv_results.items() if k.startswith('filon')}
    gsl_res   = {k: v for k, v in conv_results.items() if k.startswith('gsl')}

    fig, axes = plt.subplots(2, 2, figsize=(12, 9),
                              gridspec_kw=dict(hspace=0.35, wspace=0.3))
    (ax_f_spec, ax_g_spec), (ax_f_diff, ax_g_diff) = axes

    # Use the highest-param result of each family as the reference for relative diff
    def _ref(d):
        if not d: return None
        return d[max(d, key=lambda k: d[k]['param'])]

    filon_ref = _ref(filon_res)
    gsl_ref   = _ref(gsl_res)

    kR = (filon_ref or gsl_ref)['w'] * D

    for name, r in filon_res.items():
        ax_f_spec.plot(r['w'] * D, r['spec'], color=r['color'], lw=1.5,
                       label=r['label'])
        if filon_ref and name != filon_ref['name']:
            spec_i = np.interp(filon_ref['w'], r['w'], r['spec'])
            rd = (spec_i - filon_ref['spec']) / (np.abs(filon_ref['spec']) + 1e-30)
            ax_f_diff.plot(filon_ref['w'] * D, rd * 100,
                           color=r['color'], lw=1.5, label=r['label'])

    for name, r in gsl_res.items():
        ax_g_spec.plot(r['w'] * D, r['spec'], color=r['color'], lw=1.5,
                       label=r['label'])
        if gsl_ref and name != gsl_ref['name']:
            spec_i = np.interp(gsl_ref['w'], r['w'], r['spec'])
            rd = (spec_i - gsl_ref['spec']) / (np.abs(gsl_ref['spec']) + 1e-30)
            ax_g_diff.plot(gsl_ref['w'] * D, rd * 100,
                           color=r['color'], lw=1.5, label=r['label'])

    for ax, title in [(ax_f_spec, 'Filon — spectrum'),
                      (ax_g_spec, 'GSL — spectrum')]:
        ax.set_yscale('log'); ax.set_xscale('log')
        ax.set_ylabel(r'$\Omega_\mathrm{GW}$ [a.u.]')
        ax.set_title(title, fontsize=9)
        ax.legend(frameon=False, fontsize=7)

    ref_label = {True: f'vs N={filon_ref["param"]}', False: ''}
    for ax, title, has_ref in [
            (ax_f_diff, f'Filon rel. diff vs N={filon_ref["param"] if filon_ref else "?"}', bool(filon_ref)),
            (ax_g_diff, f'GSL rel. diff vs LIMIT={gsl_ref["param"] if gsl_ref else "?"}', bool(gsl_ref))]:
        ax.axhline(0, color='grey', lw=0.8, ls=':')
        ax.set_xscale('log')
        ax.set_xlabel(r'$kR_*$'); ax.set_ylabel('rel. diff [%]')
        ax.set_title(title, fontsize=9)
        ax.legend(frameon=False, fontsize=7)

    plt.suptitle(
        rf'Convergence study — $\bar\lambda={LAMBDA_BAR}$, $d={D}$, $\gamma={gamma:.3f}$',
        fontsize=10
    )
    plt.savefig(REPO_ROOT / f'convergence_{TAG}.pdf', bbox_inches='tight')
    plt.show()